In [1]:
import numpy as np
import statsmodels.api as sm
rng = np.random.default_rng(42)

def _fit_ols(y, X):
    """Return dict with R2, adjR2, SSE, SSR, TSS, df, p (excl. intercept)."""
    y = np.asarray(y).reshape(-1)
    n = y.shape[0]
    if X is None or X.size == 0:
        X_ = np.ones((n,1))
        p = 0
    else:
        X = np.asarray(X, float)
        # drop any constant columns in X (avoids singularities)
        X = X[:, np.std(X, axis=0, ddof=1) > 0]
        X_ = sm.add_constant(X, has_constant='add')
        p = X.shape[1]
    model = sm.OLS(y, X_).fit()
    yhat = model.fittedvalues
    resid = y - yhat
    TSS = np.sum((y - y.mean())**2)
    SSE = np.sum(resid**2)
    SSR = TSS - SSE
    R2 = 0.0 if TSS == 0 else 1 - SSE/TSS
    # Adjusted R2 (Ezekiel)
    adjR2 = 1 - (1 - R2) * (n - 1) / max(n - p - 1, 1)
    return dict(R2=R2, adjR2=adjR2, SSE=SSE, SSR=SSR, TSS=TSS, n=n, p=p)

def _freedman_lane_p(y, A, R, n_perm=999):
    """
    Permutation test for the *pure* effect of block A given reduced block(s) R.
    Returns p-value (one-sided, improvement in SSR).
    """
    # Fit reduced model y ~ R
    y = np.asarray(y).reshape(-1)
    if R is None or R.size == 0:
        R_ = None
        fit_R = _fit_ols(y, None)
        yhat_R = np.full_like(y, y.mean())
        resid_R = y - yhat_R
    else:
        R = np.asarray(R, float)
        fit_R = _fit_ols(y, R)
        # re-fit to extract fitted values and residuals
        Xr = sm.add_constant(R, has_constant='add')
        yhat_R = sm.OLS(y, Xr).fit().fittedvalues
        resid_R = y - yhat_R

    # Observed delta SSR from adding A
    XR = R if R is not None and R.size else None
    SSR_reduced = _fit_ols(y, XR)["SSR"]
    SSR_full    = _fit_ols(y, np.c_[R, A] if XR is not None else A)["SSR"]
    T_obs = SSR_full - SSR_reduced

    # Permutations (Freedman–Lane)
    ge = 1  # count observed
    for _ in range(n_perm):
        y_star = yhat_R + rng.permutation(resid_R)
        SSR_r = _fit_ols(y_star, XR)["SSR"]
        SSR_f = _fit_ols(y_star, np.c_[R, A] if XR is not None else A)["SSR"]
        if (SSR_f - SSR_r) >= T_obs - 1e-12:
            ge += 1
    return ge / (n_perm + 1)

def varpart3(y, X=None, S=None, P=None, n_perm=999):
    """
    Variation partitioning for three blocks (X, S, P) on a univariate y.
    Returns adjusted-R2 fractions and permutation p-values for pure components.
    """
    # Convenience: helper for model R2
    def adjR2(*blocks):
        B = [b for b in blocks if b is not None and b.size]
        if not B:
            return 0.0
        Bcat = np.concatenate(B, axis=1)
        return _fit_ols(y, Bcat)["adjR2"], _fit_ols(y, Bcat)["R2"]

    # Adjusted R^2 for all combinations (effect sizes)
    R2abc_adj, R2abc = adjR2(X,S,P)
    R2ab_adj,  _     = adjR2(X,S)
    R2ac_adj,  _     = adjR2(X,P)
    R2bc_adj,  _     = adjR2(S,P)
    R2a_adj,   _     = adjR2(X)
    R2b_adj,   _     = adjR2(S)
    R2c_adj,   _     = adjR2(P)

    # Pure fractions (adjusted R^2, by subtraction of partial models)
    a_pure = max(0.0, R2abc_adj - R2bc_adj)  # X | S,P
    b_pure = max(0.0, R2abc_adj - R2ac_adj)  # S | X,P
    c_pure = max(0.0, R2abc_adj - R2ab_adj)  # P | X,S

    # Shared fractions (can be slightly negative due to adjustment/collinearity)
    ab_sh  = R2ab_adj - a_pure - b_pure
    ac_sh  = R2ac_adj - a_pure - c_pure
    bc_sh  = R2bc_adj - b_pure - c_pure
    # Triple overlap, whatever remains within total
    abc_sh = R2abc_adj - (a_pure + b_pure + c_pure + ab_sh + ac_sh + bc_sh)
    unexpl = 1.0 - R2abc_adj

    # Permutation p-values for *pure* effects (unadjusted SSR improvement)
    p_X = _freedman_lane_p(y, A=X, R=np.c_[S,P] if (S is not None and P is not None) else (S if P is None else P),
                           n_perm=n_perm) if X is not None and X.size else np.nan
    p_S = _freedman_lane_p(y, A=S, R=np.c_[X,P] if (X is not None and P is not None) else (X if P is None else P),
                           n_perm=n_perm) if S is not None and S.size else np.nan
    p_P = _freedman_lane_p(y, A=P, R=np.c_[X,S] if (X is not None and S is not None) else (X if S is None else S),
                           n_perm=n_perm) if P is not None and P.size else np.nan

    return {
        "fractions_adjR2": {
            "pure_X": a_pure, "pure_S": b_pure, "pure_P": c_pure,
            "shared_XS": ab_sh, "shared_XP": ac_sh, "shared_SP": bc_sh,
            "shared_XSP": abc_sh, "unexplained": unexpl, "total_adjR2": R2abc_adj
        },
        "pvals_pure": {"X|SP": p_X, "S|XP": p_S, "P|XS": p_P}
    }


In [ ]:
# y: (n,) vector of your trait (e.g., caffeine_%)
# X_env: (n, p1) environmental features (scaled)
# E_mem: (n, p2) selected spatial MEMs
# P_phy: (n, p3) selected phylogenetic eigenvectors

res = varpart3(y, X=X_env, S=E_mem, P=P_phy, n_perm=999)
res["fractions_adjR2"], res["pvals_pure"]
